# BERT Pre-Fine-Tuning on OWS Unlabeled Corpus (Masked Language Modeling)

This notebook performs domain-adaptive pre-training (masked language modeling, MLM) of BERT models
on large-scale unlabeled web texts from the OpenWebSearch (OWS) corpus.

**Pre-fine-tuning** means continuing BERT's masked language modeling objective on in-domain data
before task-specific fine-tuning on labeled hate speech datasets. This step is shown to improve
downstream classification performance (Gururangan et al., 2020).

**Four model variants are trained**, each on a different language subset of OWS:
- `OwsEng` — English texts
- `OwsDeu` — German texts
- `OwsSpa` — Spanish texts
- `Ows4L` — All four languages combined (2.7M texts)

**Input datasets** (loaded from Hugging Face):
- `danghaidang-passau/2_7M_texts` — 4-language combined corpus
- `danghaidang-passau/eng_unlabel`, `deu_unlabel`, `span_unlabel` — individual language corpora

**Output**: Pre-fine-tuned BERT checkpoints saved locally and uploaded to Hugging Face.

In [4]:
import pandas as pd
import json
import os
import numpy as np
from datasets import Dataset

from transformers import (
    set_seed,
    TrainingArguments, 
    Trainer,
    DataCollatorForLanguageModeling,
    AutoModelForMaskedLM,
    AutoTokenizer,
)

from time import time
import pickle
import matplotlib.pyplot as plt
import random
from tqdm import tqdm
# from unsloth import FastLanguageModel
import torch

def_seed = 42

set_seed(def_seed)
np.random.seed(def_seed)
import random
random.seed(def_seed)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
from datasets import load_dataset
# from trl import SFTTrainer, SFTConfig

from sklearn.model_selection import train_test_split


## 2. Define MLM Pre-Fine-Tuning Function

**`pretrain_Bert`**: Implements masked language modeling (MLM) continuation pre-training for a BERT model.
The function:
1. Tokenizes and chunks the input dataframe texts into 512-token windows
2. Applies `DataCollatorForLanguageModeling` with 15% random token masking
3. Wraps Hugging Face `Trainer` with configurable training arguments
4. Saves the resulting checkpoint to `save_path`

This function is called once per language variant (OwsEng, OwsDeu, OwsSpa, Ows4L).

In [ ]:

def pretrain_Bert(
        df=None,
        base=None,
        tokenizer=None,
        save_path=None,
):
    
    df = df.sample(frac=1, random_state=def_seed).reset_index(drop=True)

    dataset = Dataset.from_pandas(df)
    
    def tokenize_and_chunk(example):
        outputs = tokenizer(
            example["text"],
            truncation=True,
            max_length=512,
            stride=0,  
        )

        result = {
            "input_ids": outputs["input_ids"],
            "attention_mask": outputs["attention_mask"],
        }

        return result


    tokenized_dataset = dataset.map(
        tokenize_and_chunk,
        batched=True,
        remove_columns=["text"],  
        num_proc=10,
    )
    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=True,
        mlm_probability=0.15,
    )

    training_args = TrainingArguments(
        output_dir=save_path,
        overwrite_output_dir=True,
        num_train_epochs=1,
        save_strategy="steps",        
        save_steps=3000,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=1,
        learning_rate=1e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.2,
        max_grad_norm=0.2,
        logging_steps=200,
        fp16=False,
        bf16=True,
        gradient_accumulation_steps=2,
        gradient_checkpointing=True,
        report_to="none",              
        eval_steps=5000,
        max_steps=-1,                  
        log_level="debug",
        save_total_limit=10,
        dataloader_num_workers=16,
    )



    trainer = Trainer(
        model=base,
        args=training_args,
        train_dataset=tokenized_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator,

    )
    # trainer.train()
    trainer.train(resume_from_checkpoint=True)
    return trainer

## 3. Load OWS Unlabeled Datasets from Hugging Face

Load the four OWS unlabeled text corpora from Hugging Face. Each dataset was collected from
OpenWebSearch and filtered via URL/schema.org metadata to target conversational web content:
- `danghaidang-passau/2_7M_texts` — combined 4-language corpus (~2.7M texts)
- `danghaidang-passau/eng_unlabel` — English-only subset
- `danghaidang-passau/deu_unlabel` — German-only subset
- `danghaidang-passau/span_unlabel` — Spanish-only subset

Select the desired dataset in the next cell to build the training dataframe.

In [106]:
dataset_lgb = load_dataset("danghaidang-passau/lgb_Data")


In [115]:
from datasets import load_dataset

ds_lgb = load_dataset(repo, "LightGBM_dataset")       
ds_lgb = ds_lgb["train"]

LightGBM_dataset/lgb_df.parquet:   0%|          | 0.00/4.19M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/48598 [00:00<?, ? examples/s]

In [120]:
ds_lgb = ds_lgb.to_pandas()

In [122]:
ds_lgb.loc[ds_lgb['ds'] == 'HateSpeechX', 'ds'] = "HateXplain"

In [12]:
lgb_df  = pd.DataFrame(dataset_lgb['train'])
# lgb_df = pd.read_csv('lgb_data_2_label.csv')
lgb_df['ds'].value_counts()

ds
HateSpeechX     15299
Sexism          10904
GermEval2019     9698
ViHSD            8061
GermEval2021     2071
US_election      1283
Covid            1282
Name: count, dtype: int64

In [108]:
dataset_lgb['train'].to_pandas()['ds'].value_counts()

ds
HateSpeechX     15299
Sexism          10904
GermEval2019     9698
ViHSD            8061
GermEval2021     2071
US_election      1283
Covid            1282
Name: count, dtype: int64

Index(['index', 'dataset', 'multi_label_name', 'multi_label_id', 'text',
       'hate_label_id', 'language', 'len_text', 'prompt',
       'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1',
       'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2',
       'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_1',
       'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_2',
       'unsloth/gemma-2-9b-it-bnb-4bit_label_1',
       'unsloth/gemma-2-9b-it-bnb-4bit_label_2',
       'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1',
       'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2', 'ds', 'label_id'],
      dtype='object')

In [104]:
dataset = load_dataset("danghaidang-passau/HateOWS-human-dataset")

In [105]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'language', 'ds', 'label_id'],
        num_rows: 126475
    })
    test: Dataset({
        features: ['text', 'language', 'ds', 'label_id'],
        num_rows: 31365
    })
})

In [89]:
df_eng = dataset['spa'].to_pandas()

In [90]:
df_eng = df_eng.dropna(axis=1, how="all")


In [92]:
df_eng.shape

(1085275, 3)

In [62]:
df.shape

(240647, 17)

In [103]:
dataset_human = load_dataset("danghaidang-passau/HateOWS-human-dataset")

In [29]:
df_human = pd.DataFrame(dataset_human['train'])

In [102]:
dataset_human['test'].to_pandas()['ds'].value_counts()

ds
HateXplain      3846
GermEval18      3532
AHSD            3000
ViHSD           2672
Sexism          2632
GermEval19      2507
Gahd            2198
GermEval21      2085
Chileno         1928
HateEval-spa    1286
Haternet        1205
US_election     1117
HateEval-eng    1000
Covid            971
AbusEval         860
HASOC            526
Name: count, dtype: int64

In [99]:
df_human.ds.value_counts()

ds
AHSD            21783
HateXplain      15299
AbusEval        13240
Sexism          10904
GermEval19       9698
HateEval-eng     9000
Gahd             8797
ViHSD            8061
Chileno          7572
HateEval-spa     5309
GermEval18       5009
Haternet         4794
HASOC            2373
GermEval21       2071
US_election      1283
Covid            1282
Name: count, dtype: int64

In [9]:
df = pd.DataFrame(dataset['annotated'])

In [14]:
lgb_df['qwen_prob_1'] = lgb_df['unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1']
del lgb_df['unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1']

In [26]:
lgb_df['mistral_prob_2'] = lgb_df['mean_prob_2']
del lgb_df['mean_prob_2']

In [76]:
df_4L.shape

(2786912, 3)

In [5]:
import pickle
with open("/home/dangdang/Downloads/latex/HateLLmOWS2-main/Bert_report.pkl", "rb") as f:
    model_probs_dict = pickle.load(f)

In [23]:
with open("/home/dangdang/Downloads/latex/HateLLmOWS2-main/Evaluation/Bert_probs.pkl", "wb") as f:
    pickle.dump(model_probs_dict, f)

In [22]:
for k,v in model_probs_dict.items():
    print(v.keys())

dict_keys(['OWS_spa', 'Bert', 'Hate'])
dict_keys(['OWS_deu', 'Bert', 'Hate'])
dict_keys(['Bert', 'Hate', 'OWS_eng'])
dict_keys(['Bert', 'Hate', 'OWS_4L'])
dict_keys(['Bert', 'Hate', 'OWS_4L'])
dict_keys(['Bert', 'Hate', 'OWS_4L'])


In [20]:
model_probs_dict['7set']['OWS_4L'] = model_probs_dict['7set']['OWS_spa']

In [21]:
del model_probs_dict['7set']['OWS_spa']

In [123]:
import io
import os
import pyarrow as pa
import pyarrow.parquet as pq
from huggingface_hub import HfApi

repo_id = "danghaidang-passau/HateOWS-dataset-LREC2026"

# prepare DataFrame (you already have lgb_df)
df = ds_lgb.reset_index(drop=True)

# convert to Arrow Table and write parquet to in-memory buffer
table = pa.Table.from_pandas(df)
buf = io.BytesIO()
pq.write_table(table, buf, compression="snappy")
buf.seek(0)

api = HfApi()
api.upload_file(
    path_or_fileobj=buf,
    path_in_repo="LightGBM_dataset/lgb_df.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.
[huggingface_hub._commit_api|WARNING]Uploading files as a binary IO buffer is not supported by Xet Storage. Falling back to HTTP upload.


CommitInfo(commit_url='https://huggingface.co/datasets/danghaidang-passau/HateOWS-dataset-LREC2026/commit/5208200d8f358094329407507ceb42320a1edce9', commit_message='Upload LightGBM_dataset/lgb_df.parquet with huggingface_hub', commit_description='', oid='5208200d8f358094329407507ceb42320a1edce9', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/danghaidang-passau/HateOWS-dataset-LREC2026', endpoint='https://huggingface.co', repo_type='dataset', repo_id='danghaidang-passau/HateOWS-dataset-LREC2026'), pr_revision=None, pr_num=None)

In [10]:
df.columns 

Index(['text', 'language', 'token_len', 'qwen_prob_1', 'qwen_prob_2',
       'gemma_prob_1', 'gemma_prob_2', 'llama_prob_1', 'llama_prob_2',
       'mistral_prob_1', 'mistral_prob_2', 'mean_prob_1', 'mean_prob_2',
       'mean_label', 'lgb_prob_1', 'lgb_prob_2', 'lgb_label', 'vote_label'],
      dtype='object')

In [13]:
datataset_4L_2_7M = load_dataset("danghaidang-passau/2_7M_texts")
datataset_Eng = load_dataset("danghaidang-passau/eng_unlabel")
datataset_Deu = load_dataset("danghaidang-passau/deu_unlabel")
# datataset_Spa = load_dataset("danghaidang-passau/span_unlabel")


In [16]:
df = pd.DataFrame(datataset_4L_2_7M['train'])

In [39]:
df.language.value_counts()

language
eng    1200000
deu     900000
spa     500000
vie     186912
Name: count, dtype: int64

In [41]:
df.iloc[0]

text           La idea es romántica…pero para pagarles ese pa...
language                                                     spa
len_token                                                     38
y_pred_bert                                                    0
Name: 0, dtype: object

In [22]:
dataset = load_dataset("danghaidang-passau/df_eval_16")

Generating train split: 100%|██████████| 31365/31365 [00:00<00:00, 740679.71 examples/s]


In [26]:
dataset_2label_with_probs = load_dataset("danghaidang-passau/Hate.2_labels_labeled")
datataset_4L_2_7M = load_dataset("danghaidang-passau/2_7M_texts")


Generating 2_labels split: 100%|██████████| 240647/240647 [00:00<00:00, 1044407.43 examples/s]


In [27]:
dataset_2label_with_probs

DatasetDict({
    2_labels: Dataset({
        features: ['text', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2', 'unsloth/gemma-2-9b-it-bnb-4bit_label_1', 'unsloth/gemma-2-9b-it-bnb-4bit_label_2', 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_1', 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_2', 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1', 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2', 'language', 'mean_label_1', 'mean_label_2', 'Mean', 'lgb_label_1', 'lgb_label_2', 'Lgb', 'Vote'],
        num_rows: 240647
    })
})

In [25]:
df['language'].value_counts()

language
eng    1200000
deu     900000
spa     500000
vie     186912
Name: count, dtype: int64

In [23]:
df_eval = pd.DataFrame(dataset['train'])

In [18]:
df.iloc[0]

text           La idea es romántica…pero para pagarles ese pa...
language                                                     spa
len_token                                                     38
y_pred_bert                                                    0
Name: 0, dtype: object

In [12]:
datataset_4L_2_7M = load_dataset("danghaidang-passau/2_7M_texts")
datataset_Eng = load_dataset("danghaidang-passau/eng_unlabel")
datataset_Deu = load_dataset("danghaidang-passau/deu_unlabel")
datataset_Spa = load_dataset("danghaidang-passau/span_unlabel")


DatasetNotFoundError: Dataset 'danghaidang-passau/span_unlabel' doesn't exist on the Hub or cannot be accessed.

## 4. Build Training DataFrame

Convert the loaded Hugging Face dataset split to a pandas DataFrame. Use the `['train']` split which
contains all texts for that language variant. Change the dataset variable to switch between language
configurations (e.g., `datataset_Eng`, `datataset_Deu`, `datataset_Spa`, or `datataset_4L_2_7M`).

In [ ]:
df = pd.DataFrame(datataset_4L_2_7M['train'])

## 5. Load BERT Base Model and Tokenizer

Load the base BERT model and its tokenizer from Hugging Face. The base is `google-bert/bert-base-uncased`
for English, German, and multilingual variants, or a language-specific BERT for other configurations.
The tokenizer is used for MLM tokenization inside `pretrain_Bert`.

In [ ]:
model_name = "google-bert/bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

## 6. Run Pre-Fine-Tuning

Call `pretrain_Bert` to start MLM pre-fine-tuning on the selected OWS corpus. The trained checkpoint
will be saved to the `save_path` directory (e.g., `"Ows4L"`). Training time depends on corpus size;
the 4-language 2.7M-text corpus takes approximately 6-8 hours on a single A100 GPU.

After training, upload the checkpoint to Hugging Face (`danghaidang-passau/Ows4L_16`, etc.) for use
in `eval_LLMs.ipynb`.

In [ ]:
trainer = pretrain_Bert(
    df=df,
    base=model,
    tokenizer=tokenizer,
    save_path="Ows4L"
)

## 7. Build and Publish LREC 2026 OWS Dataset Repo

This section publishes the dataset used in the LREC 2026 paper:

**Toward Generalized Cross-Lingual Hateful Language Detection with Web-Scale Data and Ensemble LLM Annotations**

### What this cell does

- Combines OWS raw corpora into explicit language splits
- Renames `len_token` to `token_len` for consistency
- Normalizes and shortens annotation column names
- Publishes to one Hugging Face dataset repo
- Generates a README with:
  - row counts and token totals by language
  - hate-count summary for 4 annotator LLMs + 3 ensemble methods

### Final split names

- `deu_eng_spa_vie` (4-language OWS corpus from `2_7M_texts`)
- `deu`
- `eng`
- `spa`
- `annotated`

In [47]:
from datasets import load_dataset
import os

# ---------- CONFIG ----------
HF_USERNAME = "danghaidang-passau"  # change if needed
REPO_NAME = "HateOWS-dataset-LREC2026"
REPO_ID = f"{HF_USERNAME}/{REPO_NAME}"
PRIVATE = False

RAW_4L_DATASET_ID = "danghaidang-passau/2_7M_texts"
RAW_DEU_DATASET_ID = "danghaidang-passau/deu_unlabel"
RAW_ENG_DATASET_ID = "danghaidang-passau/eng_unlabel"
RAW_SPA_DATASET_ID = "danghaidang-passau/spa_unlabel"  # fallback handled in publish cell
RAW_SPA_FALLBACK_DATASET_ID = "danghaidang-passau/span_unlabel"
ANNOTATED_DATASET_ID = "danghaidang-passau/Hate.2_labels_labeled"

HF_TOKEN = os.getenv("HF_TOKEN", None)

print("Target repo:", REPO_ID)

Target repo: danghaidang-passau/HateOWS-dataset-LREC2026


In [34]:
ann_ds

DatasetDict({
    2_labels: Dataset({
        features: ['text', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1', 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2', 'unsloth/gemma-2-9b-it-bnb-4bit_label_1', 'unsloth/gemma-2-9b-it-bnb-4bit_label_2', 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_1', 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_2', 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1', 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2', 'language', 'mean_label_1', 'mean_label_2', 'Mean', 'lgb_label_1', 'lgb_label_2', 'Lgb', 'Vote'],
        num_rows: 240647
    })
})

In [ ]:
# Deprecated: old upload flow kept only for history.
# Use Cell 26 for the current LREC 2026 dataset publishing pipeline.

ValueError: All datasets in `DatasetDict` should have the same features but features for 'raw' and 'annotated' don't match: {'text': Value('string'), 'language': Value('string'), 'len_token': Value('int64'), 'y_pred_bert': Value('int64')} != {'text': Value('string'), 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1': Value('float64'), 'unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2': Value('float64'), 'unsloth/gemma-2-9b-it-bnb-4bit_label_1': Value('float64'), 'unsloth/gemma-2-9b-it-bnb-4bit_label_2': Value('float64'), 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_1': Value('float64'), 'unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_2': Value('float64'), 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1': Value('float64'), 'unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2': Value('float64'), 'language': Value('string'), 'mean_label_1': Value('float64'), 'mean_label_2': Value('float64'), 'Mean': Value('int64'), 'lgb_label_1': Value('float64'), 'lgb_label_2': Value('float64'), 'Lg

In [3]:
df_eval = load_dataset("danghaidang-passau/df_eval_16")
df_train = load_dataset("danghaidang-passau/train_set")


In [5]:
df_eval = pd.DataFrame(df_eval['train'])
df_train = pd.DataFrame(df_train['train'])

In [8]:
df_train.iloc[0]

text        @dualipuhs Leave my ship alone you whore https...
language                                                  eng
ds                                               HateEval-eng
label_id                                                    1
Name: 0, dtype: object

In [11]:
df_eval.label_id.value_counts()

label_id
2    19966
1    11399
Name: count, dtype: int64

In [14]:
df_eval.drop(columns=['y_ture', 'y_pred', 'prompt', 'token_len', 'probs'], inplace=True)

## 9. Publish `HateOWS-human-dataset` to Hugging Face

Combines the 16-dataset human-labelled evaluation split (`df_eval_16`) and the training split (`train_set`)
into a single unified HuggingFace dataset repo: **`danghaidang-passau/HateOWS-human-dataset`**.

### Splits published
- `train` — all training instances across 16 datasets
- `test`  — all test/evaluation instances across 16 datasets

### Datasets included (16 total, 4 languages)
| Dataset | Lang | Train | Test |
|---------|------|------:|-----:|
| HateXplain | eng | 15,299 | 3,846 |
| Sexism | eng | 10,904 | 2,632 |
| Covid | eng | 1,282 | 971 |
| US_election | eng | 1,283 | 1,117 |
| HateEval-eng | eng | 9,000 | 1,000 |
| AbusEval | eng | 13,240 | 860 |
| AHSD | eng | 21,783 | 3,000 |
| GermEval21 | deu | 2,071 | 2,085 |
| GermEval19 | deu | 9,698 | 2,507 |
| GermEval18 | deu | 5,009 | 3,532 |
| HASOC | deu | 2,373 | 526 |
| Gahd | deu | 8,797 | 2,198 |
| ViHSD | vie | 8,061 | 2,672 |
| Haternet | spa | 4,794 | 1,205 |
| HateEval-spa | spa | 5,309 | 1,286 |
| Chileno | spa | 7,572 | 1,928 |


In [16]:
import textwrap
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi
import os
# ── CONFIG ────────────────────────────────────────────────────────────────────
HF_USERNAME  = "danghaidang-passau"
REPO_NAME    = "HateOWS-human-dataset"
REPO_ID      = f"{HF_USERNAME}/{REPO_NAME}"
PRIVATE      = False
HF_TOKEN     = os.getenv("HF_TOKEN", None)

In [17]:
HF_TOKEN

In [18]:
import textwrap
import pandas as pd
from datasets import Dataset, DatasetDict
from huggingface_hub import HfApi
import os
# ── CONFIG ────────────────────────────────────────────────────────────────────
HF_USERNAME  = "danghaidang-passau"
REPO_NAME    = "HateOWS-human-dataset"
REPO_ID      = f"{HF_USERNAME}/{REPO_NAME}"
PRIVATE      = False
HF_TOKEN     = os.getenv("HF_TOKEN", None)   # set env var or replace with token string

# ── DATASET METADATA (from paper Table 1) ────────────────────────────────────
DATASET_INFO = [
    # (name,          lang,  train_n, train_hate_pct, test_n, test_hate_pct, in_7set, reference)
    ("HateXplain",   "eng",  15299, 0.59,  3846, 0.59, True,  "Mathew et al. (2022)"),
    ("Sexism",       "eng",  10904, 0.13,  2632, 0.13, True,  "Samory et al. (2021)"),
    ("Covid",        "eng",   1282, 0.19,   971, 0.20, True,  "COVID-19 dataset"),
    ("US_election",  "eng",   1283, 0.12,  1117, 0.13, True,  "Grimminger & Klinger (2021)"),
    ("HateEval-eng", "eng",   9000, 0.42,  1000, 0.43, False, "Basile et al. (2019)"),
    ("AbusEval",     "eng",  13240, 0.21,   860, 0.21, False, "Zampieri et al. (2019)"),
    ("AHSD",         "eng",  21783, 0.83,  3000, 0.83, True,  "Davidson et al. (2017)"),
    ("GermEval21",   "deu",   2071, 0.33,  2085, 0.37, True,  "Risch et al. (2021)"),
    ("GermEval19",   "deu",   9698, 0.33,  2507, 0.34, True,  "Germeval 2019"),
    ("GermEval18",   "deu",   5009, 0.34,  3532, 0.34, False, "Germeval 2018"),
    ("HASOC",        "deu",   2373, 0.28,   526, 0.25, False, "HASOC 2019"),
    ("Gahd",         "deu",   8797, 0.42,  2198, 0.43, False, "Goldzycher et al. (2024)"),
    ("ViHSD",        "vie",   8061, 0.17,  2672, 0.18, True,  "Luu et al. (2021)"),
    ("Haternet",     "spa",   4794, 0.26,  1205, 0.25, False, "Basile et al. (2019)"),
    ("HateEval-spa", "spa",   5309, 0.41,  1286, 0.43, False, "SemEval 2019"),
    ("Chileno",      "spa",   7572, 0.06,  1928, 0.06, False, "Arango-Monnar et al. (2022)"),
]

# ── BUILD DATASET SPLITS ─────────────────────────────────────────────────────
ds_train = Dataset.from_pandas(df_train.reset_index(drop=True))
ds_test  = Dataset.from_pandas(df_eval.reset_index(drop=True))

combined_ds = DatasetDict({"train": ds_train, "test": ds_test})

# ── README TABLES ─────────────────────────────────────────────────────────────
def make_split_table():
    header = "| Dataset | Language | Train # | Train %Hate | Test # | Test %Hate | 7-Set | Reference |"
    sep    = "|---------|----------|--------:|------------:|-------:|-----------:|:-----:|-----------|"
    rows   = []
    for name, lang, trn, trn_h, tst, tst_h, s7, ref in DATASET_INFO:
        star = "✓" if s7 else ""
        rows.append(
            f"| {name} | {lang} | {trn:,} | {trn_h:.0%} | {tst:,} | {tst_h:.0%} | {star} | {ref} |"
        )
    return "\n".join([header, sep] + rows)

def make_lang_summary():
    from collections import defaultdict
    agg = defaultdict(lambda: [0, 0])
    for _, lang, trn, _, tst, _, _, _ in DATASET_INFO:
        agg[lang][0] += trn
        agg[lang][1] += tst
    header = "| Language | # Datasets | Train total | Test total |"
    sep    = "|----------|:----------:|------------:|-----------:|"
    counts = {"eng": 7, "deu": 5, "vie": 1, "spa": 3}
    rows   = [f"| {lg} | {counts[lg]} | {agg[lg][0]:,} | {agg[lg][1]:,} |" for lg in ("eng","deu","vie","spa")]
    return "\n".join([header, sep] + rows)

split_table   = make_split_table()
lang_summary  = make_lang_summary()

readme_text = textwrap.dedent(f"""
---
language:
- en
- de
- es
- vi
license: other
multilinguality: multilingual
pretty_name: HateOWS Human-Labelled Dataset (LREC 2026)
task_categories:
- text-classification
size_categories:
- 10K<n<100K
tags:
- hate-speech
- offensive-language
- multilingual
---

# HateOWS Human-Labelled Dataset

This dataset is the human-annotated benchmark used in the paper:

> **Toward Generalized Cross-Lingual Hateful Language Detection with Web-Scale Data
> and Ensemble LLM Annotations** (LREC-COLING 2026).

It consolidates **16 publicly available hate-speech datasets** across four languages
(English, German, Vietnamese, Spanish) into a single, unified binary-label resource.

---

## Splits

| Split   | Description                                  | Rows     |
|---------|----------------------------------------------|----------|
| `train` | Training instances from all 16 datasets      | {len(ds_train):,} |
| `test`  | Held-out test instances from all 16 datasets | {len(ds_test):,} |

---

## Language summary

{lang_summary}

---

## Per-dataset statistics

Inst# = number of instances; %Hate = fraction of instances labelled hateful (binary).
"7-Set ✓" marks the seven datasets used in the smaller training configuration
(Section 3.2 of the paper); Spanish datasets are excluded from this set.

{split_table}

---

## Label mapping

All annotations were mapped to a unified binary scheme: **Hate** (1) / **Neutral** (0).

- *HateXplain*, *AHSD*, *ViHSD*: original "Hate" and "Offensive" categories merged → **Hate**.
- *AbusEval*: "Implicit Abusive" and "Explicit Abusive" merged → **Hate**.
- All other datasets were already binary.

> ⚠️ Merging qualitatively different categories (hate, offensiveness, abuse) into one
> positive class is a simplification. It may inflate recall on datasets whose original
> positive class was broader (e.g., *AbusEval*). See paper §3.2 for details.

---

## Appendix: Detailed Dataset Descriptions

### English datasets (7 datasets)

| Dataset | Full name | Source | Topic | Notes |
|---------|-----------|--------|-------|-------|
| **HateXplain** | HateXplain | Twitter/Reddit (Mathew et al., 2022) | General hate speech | Tri-class (hate/offensive/normal) → hate+offensive merged; includes rationale annotations |
| **Sexism** | Call Me Sexist, But… | Twitter (Samory et al., 2021) | Sexism | Binary; 13% positive rate |
| **Covid** | COVID-19 Hate Speech | Twitter (2020) | COVID-19 xenophobia | Binary; low positive rate (19–20%) |
| **US_election** | US Election Hate | Twitter (Grimminger & Klinger, 2021) | US 2020 election | Binary; 12–13% positive |
| **HateEval-eng** | SemEval-2019 Task 5 (English) | Twitter (Basile et al., 2019) | Hate towards women/immigrants | Binary; 42–43% positive |
| **AbusEval** | AbusEval / OffComEval | Twitter (Zampieri et al., 2019) | Offensive language | Multi-class → all abusive merged |
| **AHSD** | Automated Hate Speech Detection | Twitter (Davidson et al., 2017) | Hate vs offensive vs neither | Tri-class → hate+offensive merged; 83% positive after merge |

### German datasets (5 datasets)

| Dataset | Full name | Source | Topic | Notes |
|---------|-----------|--------|-------|-------|
| **GermEval21** | GermEval 2021 | Twitter (Risch et al., 2021) | Toxic comments | Binary; ~33–37% positive |
| **GermEval19** | GermEval 2018/19 | Twitter | Offensive language | Binary; ~33–34% positive |
| **GermEval18** | GermEval 2018 | Twitter | Offensive language | Binary; ~34% positive |
| **HASOC** | HASOC 2019 (German) | Twitter/Facebook (Mandl et al., 2019) | Hate and offensive content | Binary; 25–28% positive |
| **Gahd** | GAHD | Twitter (Goldzycher et al., 2024) | German anti-hate data | Binary; ~42–43% positive |

### Vietnamese dataset (1 dataset)

| Dataset | Full name | Source | Topic | Notes |
|---------|-----------|--------|-------|-------|
| **ViHSD** | Vietnamese Hate Speech Detection | Social media (Luu et al., 2021) | General hate speech | Tri-class (hate/offensive/clean) → hate+offensive merged; 17–18% positive |

### Spanish datasets (3 datasets)

| Dataset | Full name | Source | Topic | Notes |
|---------|-----------|--------|-------|-------|
| **Haternet** | HaterNet | Twitter (Basile et al., 2019) | General hate speech | Binary; 25–26% positive; SemEval-2019 Task 5 (Spanish) |
| **HateEval-spa** | SemEval-2019 Task 5 (Spanish) | Twitter | Hate towards women/immigrants | Binary; 41–43% positive |
| **Chileno** | Chileno Offensive Language | Chilean Twitter (Arango-Monnar et al., 2022) | Offensive language | Binary; very low positive rate (~6%) |

---

## Training configurations (from paper)

| Config | Datasets included | Total train rows |
|--------|-------------------|-----------------|
| **7-Set** | HateXplain, Sexism, Covid, US_election, AHSD, GermEval21, GermEval19, ViHSD | ~{sum(r[2] for r in DATASET_INFO if r[6]):,} |
| **Eng** | 7 English datasets | ~{sum(r[2] for r in DATASET_INFO if r[1]=="eng"):,} |
| **Deu** | 5 German datasets | ~{sum(r[2] for r in DATASET_INFO if r[1]=="deu"):,} |
| **Spa** | 3 Spanish datasets | ~{sum(r[2] for r in DATASET_INFO if r[1]=="spa"):,} |
| **16-Mix** | All 16 datasets | ~{sum(r[2] for r in DATASET_INFO):,} |

---

## Citation

If you use this dataset, please cite:

```bibtex
@inproceedings{{dang2026hateows,
  title     = {{Toward Generalized Cross-Lingual Hateful Language Detection with Web-Scale Data and Ensemble LLM Annotations}},
  author    = {{Dang, Hai-Dang and Mitrović, Jelena and Granitzer, Michael}},
  booktitle = {{Proceedings of LREC-COLING 2026}},
  year      = {{2026}},
  address   = {{Torino, Italy}},
}}
```

Please also cite the individual source datasets listed in the per-dataset table above.

---

## License

Each constituent dataset retains its original license. The unified splits and label
mapping contributed by this work are released under **CC BY 4.0**.
""").strip()

# ── PUSH TO HUB ───────────────────────────────────────────────────────────────
combined_ds.push_to_hub(REPO_ID, private=PRIVATE, token=HF_TOKEN)

api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=readme_text.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
)

print(f"✓  Dataset pushed to: https://huggingface.co/datasets/{REPO_ID}")
print(f"   Train rows : {len(ds_train):,}")
print(f"   Test  rows : {len(ds_test):,}")


Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 12.51ba/s]
Processing Files (1 / 1): 100%|██████████| 10.9MB / 10.9MB, 4.93MB/s  
New Data Upload: 100%|██████████| 10.9MB / 10.9MB, 4.93MB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 37.97ba/s]
Processing Files (1 / 1): 100%|██████████| 3.10MB / 3.10MB,  795kB/s  
New Data Upload: 100%|██████████| 3.10MB / 3.10MB,  795kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.17s/ shards]


✓  Dataset pushed to: https://huggingface.co/datasets/danghaidang-passau/HateOWS-human-dataset
   Train rows : 126,475
   Test  rows : 31,365


In [48]:
from datasets import load_dataset, Features, Value, DatasetDict
from huggingface_hub import HfApi
import pandas as pd
import textwrap


def get_split(ds_dict, preferred=None):
    if preferred and preferred in ds_dict:
        return ds_dict[preferred]
    return ds_dict[list(ds_dict.keys())[0]]


def load_with_fallback(primary_id, fallback_id=None):
    try:
        return load_dataset(primary_id)
    except Exception:
        if fallback_id is None:
            raise
        return load_dataset(fallback_id)


def rename_if_exists(ds, rename_map):
    cols = set(ds.column_names)
    for old, new in rename_map.items():
        if old in cols and new not in cols:
            ds = ds.rename_column(old, new)
            cols.remove(old)
            cols.add(new)
    return ds


def align_to_unified_schema(ds, unified_features, fill_defaults):
    target_cols = list(unified_features.keys())

    for col in target_cols:
        if col not in ds.column_names:
            ds = ds.add_column(col, [fill_defaults[col]] * len(ds))

    extra_cols = [c for c in ds.column_names if c not in target_cols]
    if extra_cols:
        ds = ds.remove_columns(extra_cols)

    ds = ds.select_columns(target_cols)
    ds = ds.cast(unified_features)
    return ds


def language_stats_table(ds, split_name):
    cols = ds.column_names
    has_language = "language" in cols
    has_token_len = "token_len" in cols

    keep_cols = []
    if has_language:
        keep_cols.append("language")
    if has_token_len:
        keep_cols.append("token_len")

    if keep_cols:
        df = ds.to_pandas()[keep_cols].copy()
    else:
        df = pd.DataFrame(index=range(len(ds)))

    if not has_language:
        df["language"] = None

    grp = df.groupby("language", dropna=False, as_index=False).size()
    grp = grp.rename(columns={"size": "rows"})

    if has_token_len:
        tok = df.groupby("language", dropna=False)["token_len"].sum(min_count=1).reset_index()
        tok = tok.rename(columns={"token_len": "token_len_sum"})
        out = grp.merge(tok, on="language", how="left")
    else:
        out = grp.copy()
        out["token_len_sum"] = None

    out.insert(0, "split", split_name)

    out["language"] = out["language"].astype(object)
    out.loc[out["language"].isna(), "language"] = "null"

    out["rows"] = out["rows"].astype("int64")
    out["token_len_sum"] = out["token_len_sum"].astype(object)
    out.loc[pd.isna(out["token_len_sum"]), "token_len_sum"] = "null"

    return out


def hate_count_table(ann_df):
    # Assumption: *_prob_1 corresponds to Hate probability (same ordering as original label_1/label_2 pipeline)
    ann = ann_df.copy()

    ann["qwen_hate"] = (ann["qwen_prob_1"] >= ann["qwen_prob_2"]).astype("Int64")
    ann["gemma_hate"] = (ann["gemma_prob_1"] >= ann["gemma_prob_2"]).astype("Int64")
    ann["llama_hate"] = (ann["llama_prob_1"] >= ann["llama_prob_2"]).astype("Int64")
    ann["mistral_hate"] = (ann["mistral_prob_1"] >= ann["mistral_prob_2"]).astype("Int64")

    records = []
    for col, name in [
        ("qwen_hate", "Qwen2.5-14B"),
        ("gemma_hate", "Gemma2-9B"),
        ("llama_hate", "Llama3.1-8B"),
        ("mistral_hate", "Mistral-7B"),
        ("lgb_label", "LightGBM Ensemble"),
        ("mean_label", "Mean Ensemble"),
        ("vote_label", "Vote Ensemble"),
    ]:
        total = ann[col].notna().sum()
        hate = (ann[col] == 1).sum()
        ratio = (float(hate) / float(total) * 100.0) if total > 0 else 0.0
        records.append({"model": name, "hate_count": int(hate), "total_rows": int(total), "hate_ratio_pct": round(ratio, 2)})

    return pd.DataFrame(records)


def to_markdown_table(df):
    cols = list(df.columns)
    lines = ["| " + " | ".join(cols) + " |", "|" + "|".join(["---"] * len(cols)) + "|"]
    for _, row in df.iterrows():
        vals = [str(row[c]) for c in cols]
        lines.append("| " + " | ".join(vals) + " |")
    return "\n".join(lines)


# -------- Load source datasets --------
raw_4l_ds = load_dataset(RAW_4L_DATASET_ID)
raw_deu_ds = load_dataset(RAW_DEU_DATASET_ID)
raw_eng_ds = load_dataset(RAW_ENG_DATASET_ID)
raw_spa_ds = load_with_fallback(RAW_SPA_DATASET_ID, RAW_SPA_FALLBACK_DATASET_ID)
ann_ds = load_dataset(ANNOTATED_DATASET_ID)

raw_4l = get_split(raw_4l_ds, "train")
raw_deu = get_split(raw_deu_ds, "train")
raw_eng = get_split(raw_eng_ds, "train")
raw_spa = get_split(raw_spa_ds, "train")
ann = get_split(ann_ds, "2_labels")

# -------- Normalize columns --------
# 1) For raw: len_token -> token_len (if present)
for name, ds in [("raw_4l", raw_4l), ("raw_deu", raw_deu), ("raw_eng", raw_eng), ("raw_spa", raw_spa)]:
    pass

raw_4l = rename_if_exists(raw_4l, {"len_token": "token_len"})
raw_deu = rename_if_exists(raw_deu, {"len_token": "token_len"})
raw_eng = rename_if_exists(raw_eng, {"len_token": "token_len"})
raw_spa = rename_if_exists(raw_spa, {"len_token": "token_len"})

# remove old classifier output from 2.7M raw if present
if "y_pred_bert" in raw_4l.column_names:
    raw_4l = raw_4l.remove_columns(["y_pred_bert"])

# 2) For annotation: shorten/normalize long column names
ann_rename_map = {
    "unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_1": "qwen_prob_1",
    "unsloth/Qwen2.5-14B-Instruct-bnb-4bit_label_2": "qwen_prob_2",
    "unsloth/gemma-2-9b-it-bnb-4bit_label_1": "gemma_prob_1",
    "unsloth/gemma-2-9b-it-bnb-4bit_label_2": "gemma_prob_2",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_1": "llama_prob_1",
    "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit_label_2": "llama_prob_2",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_1": "mistral_prob_1",
    "unsloth/mistral-7b-instruct-v0.3-bnb-4bit_label_2": "mistral_prob_2",
    "mean_label_1": "mean_prob_1",
    "mean_label_2": "mean_prob_2",
    "Mean": "mean_label",
    "lgb_label_1": "lgb_prob_1",
    "lgb_label_2": "lgb_prob_2",
    "Lgb": "lgb_label",
    "Vote": "vote_label",
    "len_token": "token_len",
}
ann = rename_if_exists(ann, ann_rename_map)

# -------- Unified schema (missing fields => null) --------
unified_features = Features({
    "text": Value("string"),
    "language": Value("string"),
    "token_len": Value("int64"),
    "qwen_prob_1": Value("float64"),
    "qwen_prob_2": Value("float64"),
    "gemma_prob_1": Value("float64"),
    "gemma_prob_2": Value("float64"),
    "llama_prob_1": Value("float64"),
    "llama_prob_2": Value("float64"),
    "mistral_prob_1": Value("float64"),
    "mistral_prob_2": Value("float64"),
    "mean_prob_1": Value("float64"),
    "mean_prob_2": Value("float64"),
    "mean_label": Value("int64"),
    "lgb_prob_1": Value("float64"),
    "lgb_prob_2": Value("float64"),
    "lgb_label": Value("int64"),
    "vote_label": Value("int64"),
})

fill_defaults = {k: None for k in unified_features.keys()}

raw_4l_aligned = align_to_unified_schema(raw_4l, unified_features, fill_defaults)
raw_deu_aligned = align_to_unified_schema(raw_deu, unified_features, fill_defaults)
raw_eng_aligned = align_to_unified_schema(raw_eng, unified_features, fill_defaults)
raw_spa_aligned = align_to_unified_schema(raw_spa, unified_features, fill_defaults)
ann_aligned = align_to_unified_schema(ann, unified_features, fill_defaults)

# Final split names (no old raw/raw_2_7m)
combined = DatasetDict({
    "deu_eng_spa_vie": raw_4l_aligned,
    "deu": raw_deu_aligned,
    "eng": raw_eng_aligned,
    "spa": raw_spa_aligned,
    "annotated": ann_aligned,
})

# -------- Build README tables --------
table_unlabel_3_df = pd.concat(
    [
        language_stats_table(raw_deu_aligned, "deu"),
        language_stats_table(raw_eng_aligned, "eng"),
        language_stats_table(raw_spa_aligned, "spa"),
    ],
    ignore_index=True,
)

table_4l_df = language_stats_table(raw_4l_aligned, "deu_eng_spa_vie")
ann_df = ann_aligned.to_pandas()
table_hate_counts_df = hate_count_table(ann_df)

table_unlabel_3_md = to_markdown_table(table_unlabel_3_df)
table_4l_md = to_markdown_table(table_4l_df)
table_hate_counts_md = to_markdown_table(table_hate_counts_df)

readme_text = textwrap.dedent(f"""
---
language:
- en
- de
- es
- vi
license: other
multilinguality: multilingual
pretty_name: OWS LREC2026 Raw + Ensemble Annotations
task_categories:
- text-classification
size_categories:
- 1M<n<10M
---

# OWS Data for LREC 2026

This repository contains the data resources used in the paper:

**Toward Generalized Cross-Lingual Hateful Language Detection with Web-Scale Data and Ensemble LLM Annotations** (LREC-COLING 2026).

It provides multilingual OpenWebSearch (OWS) raw corpora and an annotated subset with 4 base LLM annotators plus 3 ensemble labeling strategies.

## Splits

- `deu_eng_spa_vie`: 4-language OWS corpus (DEU, ENG, SPA, VIE)
- `deu`: German unlabeled subset
- `eng`: English unlabeled subset
- `spa`: Spanish unlabeled subset
- `annotated`: subset with model probabilities and ensemble labels

Notes:
- Column `len_token` was normalized to `token_len`.
- Missing columns are filled with `null` for schema consistency.
- Old split names like `raw` / `raw_2_7m` are not used.

## Language stats (deu, eng, spa)

{table_unlabel_3_md}

## Language stats (deu_eng_spa_vie)

{table_4l_md}

## Hate counts on `annotated`

The table below reports hate counts for four base annotator models and three ensemble methods.
For base models, hate is derived as `prob_1 >= prob_2` (same class ordering as the original pipeline).

{table_hate_counts_md}

## Annotation columns (short names)

- Base models: `qwen_prob_1`, `qwen_prob_2`, `gemma_prob_1`, `gemma_prob_2`, `llama_prob_1`, `llama_prob_2`, `mistral_prob_1`, `mistral_prob_2`
- Ensemble probs: `mean_prob_1`, `mean_prob_2`, `lgb_prob_1`, `lgb_prob_2`
- Ensemble labels: `mean_label`, `lgb_label`, `vote_label`
- Shared metadata: `text`, `language`, `token_len`
""").strip()

# -------- Upload --------
combined.push_to_hub(REPO_ID, private=PRIVATE, token=HF_TOKEN)
api = HfApi(token=HF_TOKEN)
api.upload_file(
    path_or_fileobj=readme_text.encode("utf-8"),
    path_in_repo="README.md",
    repo_id=REPO_ID,
    repo_type="dataset",
)
print(f"Uploaded dataset and README to: {REPO_ID}")

Creating parquet from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  4.30ba/s]
Processing Files (1 / 1): 100%|██████████|  176MB /  176MB, 66.6MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 5/5 [00:01<00:00,  3.97ba/s]
Processing Files (1 / 1): 100%|██████████|  192MB /  192MB,  124MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 3/3 [00:00<00:00,  3.84ba/s]
Processing Files (1 / 1): 100%|██████████|  123MB /  123MB, 8.75MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.95ba/s]
Processing Files (1 / 1): 100%|██████████|  166MB /  166MB, 80.3MB/s  
New Data Upload: |          |  0.00B /  0.00B,  0.00B/s  
Creating parquet from Arrow format: 100%|██████████| 4/4 [00:01<00:00,  3.92ba/s]
Processing Files (1 / 1): 100%|██████████|  166MB /  166MB, 80.9MB/s  
New

Uploaded dataset and README to: danghaidang-passau/HateOWS-dataset-LREC2026


## 8. Load the New Published Repo (New Splits + New Column Names)

Run this after publishing to verify the new schema.

In [45]:
# Load from your new repo
new_repo = REPO_ID

ds_all = load_dataset(new_repo)
print(ds_all)

# Example: 4-language split
df_4l = pd.DataFrame(ds_all["deu_eng_spa_vie"])
print("deu_eng_spa_vie columns:", df_4l.columns.tolist())

# Example: annotated split with short column names
df_ann = pd.DataFrame(ds_all["annotated"])
short_cols = [
    "text", "language", "token_len",
    "qwen_prob_1", "qwen_prob_2",
    "gemma_prob_1", "gemma_prob_2",
    "llama_prob_1", "llama_prob_2",
    "mistral_prob_1", "mistral_prob_2",
    "mean_prob_1", "mean_prob_2", "mean_label",
    "lgb_prob_1", "lgb_prob_2", "lgb_label", "vote_label",
]
print("annotated columns:", [c for c in short_cols if c in df_ann.columns])

# Quick check: ensemble hate counts
print("\nHate counts (1 = hate)")
for col in ["mean_label", "lgb_label", "vote_label"]:
    if col in df_ann.columns:
        print(col, int((df_ann[col] == 1).sum()))

Generating spa split: 100%|██████████| 1085275/1085275 [00:02<00:00, 441474.75 examples/s]


DatasetDict({
    annotated: Dataset({
        features: ['text', 'language', 'token_len', 'qwen_prob_1', 'qwen_prob_2', 'gemma_prob_1', 'gemma_prob_2', 'llama_prob_1', 'llama_prob_2', 'mistral_prob_1', 'mistral_prob_2', 'mean_prob_1', 'mean_prob_2', 'mean_label', 'lgb_prob_1', 'lgb_prob_2', 'lgb_label', 'vote_label'],
        num_rows: 240647
    })
    deu: Dataset({
        features: ['text', 'language', 'token_len', 'qwen_prob_1', 'qwen_prob_2', 'gemma_prob_1', 'gemma_prob_2', 'llama_prob_1', 'llama_prob_2', 'mistral_prob_1', 'mistral_prob_2', 'mean_prob_1', 'mean_prob_2', 'mean_label', 'lgb_prob_1', 'lgb_prob_2', 'lgb_label', 'vote_label'],
        num_rows: 641830
    })
    deu_eng_spa_vie: Dataset({
        features: ['text', 'language', 'token_len', 'qwen_prob_1', 'qwen_prob_2', 'gemma_prob_1', 'gemma_prob_2', 'llama_prob_1', 'llama_prob_2', 'mistral_prob_1', 'mistral_prob_2', 'mean_prob_1', 'mean_prob_2', 'mean_label', 'lgb_prob_1', 'lgb_prob_2', 'lgb_label', 'vote_label'],
 